# A. SIMPLE 3D CNN MODEL FROM Pytorch

## Bài tập 2

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
from dataset import get_dataloaders
from cnn3d_v2 import SimpleCNN3D_v2

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [4]:
data_root = "./dataset/UCF50"
batch_size = 32
num_frames = 16
img_size = 112
epochs = 50
learning_rate = 3e-4

train_loader, val_loader, class_to_idx = get_dataloaders(
    data_root=data_root,
    batch_size=batch_size,
    num_frames=num_frames,
    img_size=img_size,
    output_format="CTHW",
    split_ratio=0.8,
    num_workers=8
)

num_classes = len(class_to_idx)
print("Classes:", num_classes)
print("Train samples:", len(train_loader.dataset))
print("Val samples:", len(val_loader.dataset))

Classes: 50
Train samples: 5326
Val samples: 1355


In [5]:
torch.backends.cudnn.benchmark = True

model = SimpleCNN3D_v2(num_classes=num_classes, k1=1, k2=3, k3=5)

if torch.cuda.device_count() > 1:
    print(f"Sử dụng {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scaler = torch.amp.GradScaler('cuda')

Sử dụng 2 GPUs!


In [ ]:
history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": []
}

best_val_acc = 0.0

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    total_train = 0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == targets).sum().item()
        total_train += targets.size(0)

    epoch_train_loss = train_loss / total_train
    epoch_train_acc = train_correct / total_train

    model.eval()
    val_loss = 0.0
    val_correct = 0
    total_val = 0

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, targets)

            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == targets).sum().item()
            total_val += targets.size(0)

    epoch_val_loss = val_loss / total_val
    epoch_val_acc = val_correct / total_val

    history["train_loss"].append(epoch_train_loss)
    history["train_acc"].append(epoch_train_acc)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(epoch_val_acc)

    print(f"Epoch [{epoch+1:02d}/{epochs:02d}] "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}%")

    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        save_state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
        torch.save(save_state, "best_cnn3d_v2.pth")


Epoch [01/50] Train Loss: 3.7247 | Train Acc: 5.78% | Val Loss: 3.5144 | Val Acc: 10.04%
Epoch [02/50] Train Loss: 3.2520 | Train Acc: 14.57% | Val Loss: 2.9492 | Val Acc: 20.07%
Epoch [03/50] Train Loss: 2.8189 | Train Acc: 24.15% | Val Loss: 2.6326 | Val Acc: 26.94%
Epoch [04/50] Train Loss: 2.5925 | Train Acc: 28.37% | Val Loss: 2.4449 | Val Acc: 31.88%
Epoch [05/50] Train Loss: 2.3978 | Train Acc: 33.68% | Val Loss: 2.3249 | Val Acc: 35.06%
Epoch [06/50] Train Loss: 2.2199 | Train Acc: 37.35% | Val Loss: 2.1833 | Val Acc: 38.60%
Epoch [07/50] Train Loss: 2.0702 | Train Acc: 41.36% | Val Loss: 2.0216 | Val Acc: 42.80%
Epoch [08/50] Train Loss: 1.9535 | Train Acc: 43.88% | Val Loss: 2.0109 | Val Acc: 42.73%
Epoch [09/50] Train Loss: 1.8319 | Train Acc: 47.48% | Val Loss: 1.8996 | Val Acc: 45.17%
Epoch [10/50] Train Loss: 1.7351 | Train Acc: 49.27% | Val Loss: 1.7761 | Val Acc: 49.67%
Epoch [11/50] Train Loss: 1.6102 | Train Acc: 52.55% | Val Loss: 1.7872 | Val Acc: 50.11%
Epoch [12/5

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["train_acc"], label="Train Acc")
plt.plot(history["val_acc"], label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.savefig("loss_acc_curve_v2.png")
plt.show()